<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/chong/notebooks/gsm8k/Experiment_2_Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>


# MEDAL / AB-MEDAL with IG or AdaLook — Extended Evaluation (70 Questions)

Four variants run on **all 70 frozen evaluation questions** using the E1 runner. The original 20 questions are an identical prefix; all 70 come from the GSM8K training split and the 200-question test pool is unchanged. This notebook evaluates IG versus AdaLook; the separate E2 two-step FIG runner remains available through Experiment_2_Dev and scripts/run_e2.py. Within each outer search policy, only immediate reward evaluation changes. AdaLook is the existing adaptive-lookahead hybrid from the two demos, not a reproduction of the paper's full decoder.

| Component | Shared settings |
|---|---|
| Backbone | Pinned LLaDA-8B-Instruct, bfloat16 |
| Questions | All 70 frozen GSM8K `evaluation` questions from experiment_samples_e2.json, in manifest order |
| Prompt | Existing pens / muffins two-shot decomposition |
| Search budgets | 256 and 512 sequence NFEs (A and B), including all AdaLook forwards |
| Maximum real-tree depth | 10 |
| Actions / candidates | K1=3, K2=5, minimum C=3; deepest eligible layer, rank by empirical Q, complete rank 1 |
| MEDAL policy | Existing UCB and sibling expansion; exploration constant 0.01 |
| AB-MEDAL policy | Existing Student-t GEN/CONT and priors (0, 1, 1, 0.1) |
| Completion | Existing LLaDA Confidence, T=0, 256 output positions |
| Execution | Serial, existing per-question seeds and shared completion cache |

| Variant | Outer search | Immediate reward |
|---|---|---|
| MEDAL | MEDAL | Normalized IG |
| AB-MEDAL | AB-MEDAL | Normalized IG |
| MEDAL + AdaLook | MEDAL | `adalook_adaptive` |
| AB-MEDAL + AdaLook | AB-MEDAL | `adalook_adaptive` |

AdaLook defaults match both demos: alpha=0.1, confidence threshold=0.8, maximum lookahead depth=2, variance threshold=0.01, trigger threshold=0.2, minimum masks=2. Temporary hypotheses only evaluate reward; they never enter the real tree or final completion commitments.

## Shared continuous-checkpoint protocol

Each question/variant searches continuously with boundaries at **[256, 512]**. AdaLook admits complete comparison groups and refinement rounds only when they fit before the next boundary. Therefore prefix equivalence requires the **same earlier checkpoint boundaries**; a 512 run with a boundary at 256 need not equal a 512 run without that boundary. The boundary schedule is recorded in the effective configuration.

Search NFE = real-tree NFE + temporary-comparison NFE + rollout NFE. Total logical NFE also includes final completion, even on a cache hit. Early frontier exhaustion records actual search calls, not the requested cap.

## Results and offline E3

The full evaluation produces 560 per-question rows and 8 aggregate rows. E3 reconstructs tree metrics offline and displays the existing six figures for all four variants plus an explicit search-compute breakdown. Immediate IG remains distinct from the AdaLook reward and Q values.

The existing timing convention is retained: search wall time is apportioned by budget and cached completion time is zero. These times are not independent-run latency measurements. Output uses a new four-variant directory; resume rejects incompatible saved configurations.

A fresh run evaluates all 70 questions in one stage. Only results saved in this new output directory are resumed; historical E1 results are not imported. Reports cover the entire selected pool, without 20/50 subgroups.


## 1. Verify GPU Hardware
Verify that a CUDA GPU supporting bfloat16 (recommended: L4 or A100 with $\ge 22$ GiB VRAM) is active.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime in Colab (L4 or A100), then reconnect.")
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / (1024 ** 3)
print(f"GPU: {gpu.name} | Total VRAM: {vram_gib:.1f} GiB")
if vram_gib < 20:
    raise RuntimeError("This notebook uses unquantized BF16 weights. Choose an L4 or A100 runtime with >= 22 GB VRAM; 16 GB T4 is too small.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support bfloat16 inference.")
print("✓ GPU hardware verification passed.")

## 2. Repository Setup
Clone or update the GitHub repository on branch `chong`. Supports private repository authentication via Colab Secrets or password prompt.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "chong"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
IN_COLAB = Path("/content").is_dir()
REPO_DIR = Path("/content/CSCI544-DLM") if IN_COLAB else next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/dlm_project").is_dir()
)

def run_git(args, token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *args], env=env, capture_output=True, text=True)

def remote_git(args):
    result = run_git(args)
    if result.returncode:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("GitHub token with read access to private repo CooL-Legend/CSCI544-DLM: ")
        try:
            result = run_git(args, token)
        finally:
            del token
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")

def ensure_checkout():
    def local_git(*args):
        return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout.")
        fetch_specs = local_git("config", "--get-all", "remote.origin.fetch").splitlines()
        branch_ref = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
        if not any(spec.lstrip("+") in (branch_ref, "refs/heads/*:refs/remotes/origin/*") for spec in fetch_specs):
            local_git("remote", "set-branches", "--add", "origin", BRANCH)
        remote_git(["-C", str(REPO_DIR), "fetch", "origin", branch_ref])
        exists = subprocess.run(["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"]).returncode == 0
        if exists:
            local_git("switch", BRANCH)
        else:
            local_git("switch", "--create", BRANCH, "--track", f"origin/{BRANCH}")
        local_git("merge", "--ff-only", f"origin/{BRANCH}")
    else:
        remote_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    print(f"Code ready: {REPO_DIR} | branch: {BRANCH} | commit: {local_git('rev-parse', '--short', 'HEAD')}")

if IN_COLAB:
    ensure_checkout()
else:
    print(f"Using local repository checkout: {REPO_DIR}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

## 3. Install Dependencies
Install pinned LLaDA dependencies, editable package installation, and plotting tools.

In [ ]:
import site
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements/llada-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "pandas", "seaborn", "pytest"], check=True)

for pkg in [*site.getsitepackages(), site.getusersitepackages()]:
    site.addsitedir(pkg)
print("✓ Dependencies installed and package paths loaded.")

## 4. Run Algorithmic Pre-Flight Unit Tests
Run unit tests with synthetic models to confirm checkpoint prefix equivalence, candidate selection, completion caching, and E3 offline metric accounting before loading weights.

In [ ]:
import subprocess
import sys

res = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/decoding/test_checkpoint_equivalence.py",
     "tests/decoding/test_adalook.py", "tests/decoding/test_ab_adalook.py",
     "tests/tasks/test_experiment_1.py", "tests/tasks/test_e1_four_variants.py", "tests/tasks/test_experiment_protocol.py", "tests/tasks/test_e3_analysis.py", "-v"],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(res.stdout)
if res.returncode != 0:
    print("--- PRE-FLIGHT TESTS FAILED ---")
    print(res.stderr)
    raise RuntimeError("Algorithmic tests failed. Review environment before loading weights.")
print("✓ All pre-flight algorithmic unit tests passed.")

## 5. Mount Google Drive & Configure Cache
Mount Google Drive for persistent caching of model weights and experimental artifacts across Colab sessions.

In [ ]:
from pathlib import Path
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM")
else:
    DRIVE_ROOT = REPO_DIR / ".cache/e1_e3"

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "cache/datasets")

OUTPUT_DIR = DRIVE_ROOT / "results/adalook_eval_70"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Persistent Drive Root:", DRIVE_ROOT)
print("Experiment Output Directory:", OUTPUT_DIR)

## 6. Experiment Controls & Configuration
Use the extended frozen evaluation set (70 questions), budgets [256, 512], depth 10, minimum C=3 and serial execution. Evaluate all four variants with the existing demo AdaLook defaults.


In [ ]:
from dlm_project.tasks.gsm8k.experiment_1 import (
    CHECKPOINT_BUDGETS, MAX_SEARCH_DEPTH, MIN_CANDIDATES, context
)

STAGE = "evaluation"
EVALUATION_SET = "extended_70"
NUM_SAMPLES = None  # All 70; use 2 only for a separate GPU smoke run.
RESUME = True
METHODS = ["medal", "ab_medal", "medal_adalook", "ab_medal_adalook"]
BUDGETS = list(CHECKPOINT_BUDGETS)  # [256, 512] (Conditions A & B)
MAX_DEPTH = MAX_SEARCH_DEPTH        # 10
MIN_C = MIN_CANDIDATES              # 3
BATCH_SIZE = 1                      # Serial execution strictly enforced

# Existing demo defaults; shared by both AdaLook variants.
ADALOOK_OPTIONS = dict(
    adalook_alpha=0.1, adalook_confidence_threshold=0.8,
    adalook_max_depth=2, adalook_variance_threshold=0.01,
    adalook_trigger_threshold=0.2, adalook_min_masks=2,
)

ctx = context(
    stage=STAGE,
    evaluation_set=EVALUATION_SET,
    num_samples=NUM_SAMPLES,
    adalook_options=ADALOOK_OPTIONS,
    methods=tuple(METHODS),
    budgets=tuple(BUDGETS),
    max_depth=MAX_DEPTH,
    num_candidates=MIN_C,
)

if BATCH_SIZE != 1:
    raise ValueError("This four-variant AdaLook evaluation requires BATCH_SIZE = 1.")
OUTPUT_DIR = DRIVE_ROOT / "results/adalook_eval_70"
if NUM_SAMPLES is not None:
    OUTPUT_DIR = OUTPUT_DIR / f"smoke_{NUM_SAMPLES}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR)

print(f"=== Experiment Configuration ({STAGE.upper()}) ===")
print("  Protocol:", ctx["protocol_version"])
print(f"  Stage: {ctx['stage']} ({len(ctx['examples'])} frozen questions)")
print("  Methods:", ctx["methods"])
print("  Budgets:", ctx["budgets"])
print("  AdaLook options:", ctx["adalook_options"])
print("  Max Search Depth (Lc):", ctx["max_depth"])
print("  Candidate Selection Rule: min-C =", ctx["num_candidates"])
print("  Batch Size:", BATCH_SIZE, "(strictly serial)")
print("\nFirst question preview:")
print("  ID:", ctx["examples"][0]["example_id"])
print("  Question:", ctx["examples"][0]["question"])
print("  Answer:", ctx["examples"][0].get("reference_solution", "").split("####")[-1].strip())


## 7. Cache & Load LLaDA-8B-Instruct
Load the pinned `LLaDA-8B-Instruct` model and tokenizer on CUDA in bfloat16, and prepare the tokenized evaluation prompts.

In [ ]:
import shutil
from dlm_project.models.profiles import get_profile
from dlm_project.tasks.gsm8k.experiment_1 import load_model, prepare

profile = get_profile("llada")
model_name = profile["repo_id"].split("/")[-1]
drive_model_dir = DRIVE_ROOT / "models" / model_name
runtime_model_dir = Path("/content/models") / model_name

# Fast local NVMe cache in Colab runtime if available on Drive
if IN_COLAB and drive_model_dir.is_dir() and not (runtime_model_dir / "config.json").is_file():
    print(f"Copying model from Google Drive to local fast runtime storage: {runtime_model_dir}...")
    shutil.copytree(drive_model_dir, runtime_model_dir)

MODEL_PATH = str(runtime_model_dir if runtime_model_dir.is_dir() else (drive_model_dir if drive_model_dir.is_dir() else profile["repo_id"]))
print("Using model location:", MODEL_PATH)

wrapper = load_model(ctx, model_path=MODEL_PATH, device="cuda", local_files_only=Path(MODEL_PATH).is_dir())
prepared = prepare(wrapper, ctx)

print(f"✓ Model loaded: {wrapper.identity['family']} ({wrapper.identity['repo_id']} @ {wrapper.identity['revision']})")
print(f"✓ Prepared {len(prepared)} evaluation prompts with two-shot template.")

## 8. Optional Real-Model Checkpoint and NFE Check
Disabled by default, as in the previous notebook. Enable `RUN_REAL_MODEL_CHECKS` to check all four variants on Sample 0 at small diagnostic budgets (12, 24). This checks selected states, finalized node statistics, AB posteriors, and real-tree + comparison + rollout NFE accounting. Independent diagnostic runs retain the same earlier checkpoint boundaries. The evaluation budgets remain [256, 512].


In [ ]:
RUN_REAL_MODEL_CHECKS = False
if RUN_REAL_MODEL_CHECKS:
    from dlm_project.tasks.gsm8k.experiment_1 import execute_search_checkpointed
    from dlm_project.tasks.gsm8k.e1_variants import is_adaptive
    from dlm_project.decoding import search_audit

    diagnostic_budgets = (12, 24)
    item = prepared[0]
    for method in METHODS:
        continuous, _ = execute_search_checkpointed(
            wrapper, item, ctx, method, budgets=diagnostic_budgets,
            max_depth=MAX_DEPTH, num_candidates=MIN_C)
        for budget in diagnostic_budgets:
            prefix_budgets = tuple(b for b in diagnostic_budgets if b <= budget)
            standalone, _ = execute_search_checkpointed(
                wrapper, item, ctx, method, budgets=prefix_budgets,
                max_depth=MAX_DEPTH, num_candidates=MIN_C)
            left = continuous.checkpoints[budget]
            right = standalone.checkpoints[budget]
            assert left["selected_state_hash"] == right["selected_state_hash"]
            assert left["shortlist_node_ids"] == right["shortlist_node_ids"]
            for key in ("model_calls", "selected_q", "selected_depth"):
                assert left[key] == right[key]
            assert [(v.node_id, v.visits, v.reward_sum) for v in left["nodes"]] == [
                (v.node_id, v.visits, v.reward_sum) for v in right["nodes"]]
            if is_adaptive(method):
                assert left["posteriors"] == right["posteriors"]
            audit = search_audit.from_events(
                left["trace"] + [left["selection"]], prompt_ids=item["prompt_ids"],
                mask_id=wrapper.mask_id, gen_length=ctx["common"]["generation"]["gen_length"],
                requested_depth=MAX_DEPTH, requested_budget=budget,
                search_nfes=left["model_calls"], adaptive=is_adaptive(method), shortlist_limit=MIN_C)
            metrics = search_audit.metrics(audit)
            assert sum(metrics["nfes_per_depth"].values()) + metrics["adalook_total_extra_nfes"] == left["model_calls"]
            assert metrics["uncompleted_evaluation_nfes"] == 0
            assert left["model_calls"] <= budget
        print(f"Verified {method}: shared-boundary parity and NFE accounting.")
else:
    print("Real-model diagnostic skipped; synthetic pre-flight tests cover all four variants.")


## 9. Execute Four-Variant Extended Evaluation (70 Questions)
Execute MEDAL, AB-MEDAL, MEDAL + AdaLook, and AB-MEDAL + AdaLook across the same questions at budgets 256, 512 with checkpointing, minimum-C=3 selection, completion caching, real-time per-budget progress logging, incremental per-question saving to Drive (with `os.sync()`), and automatic session resumption (`resume=True`).

In [ ]:
from time import perf_counter
from dlm_project.tasks.gsm8k.experiment_1 import run_e1_stage

print(f"Starting four-variant evaluation on {len(ctx['examples'])} questions for {METHODS} at budgets {BUDGETS}...")
t_start = perf_counter()

# Note: resume=True automatically restores completed questions from disk if your Colab session disconnected.
# Results are incrementally saved and synced to Drive after every question.
q_rows, agg_rows = run_e1_stage(
    wrapper, ctx, OUTPUT_DIR,
    methods=tuple(METHODS),
    budgets=tuple(BUDGETS),
    max_depth=MAX_DEPTH,
    num_candidates=MIN_C,
    resume=RESUME,
    batch_size=BATCH_SIZE,
)

t_total = perf_counter() - t_start
print(f"\n✓ Extended evaluation completed in {t_total:.1f}s ({t_total/60:.2f} minutes).")
print(f"✓ All tree checkpoints, completions, and logs saved to: {OUTPUT_DIR / STAGE}")


## 10. Extended Evaluation Results & Paired Comparison
Display aggregate accuracy and efficiency metrics across budgets, followed by paired per-question outcomes.

In [ ]:
import pandas as pd
from IPython.display import display

agg_file = OUTPUT_DIR / "summaries" / f"e1_{STAGE}_aggregate.csv"
if not agg_file.is_file():
    agg_file = OUTPUT_DIR / "summaries" / "e1_aggregate.csv"

df_agg = pd.read_csv(agg_file)
print("=== Table 1: E1 Aggregate Performance & Efficiency Summary ===")
display(df_agg[[
    "search_budget", "method", "sample_count", "accuracy_percent",
    "ci_95_low", "ci_95_high", "mean_total_nfe", "mean_selected_depth",
    "mean_search_wall_seconds", "mean_completion_wall_seconds", "mean_total_wall_seconds", "fallback_rate",
    "mean_tree_nfes", "mean_temporary_candidate_nfes", "mean_adalook_extra_nfes",
    "mean_adalook_total_extra_nfes", "mean_adalook_trigger_rate"
]])

pq_file = OUTPUT_DIR / "summaries" / f"e1_{STAGE}_per_question.csv"
if not pq_file.is_file():
    pq_file = OUTPUT_DIR / "summaries" / "e1_per_question.csv"

df_pq = pd.read_csv(pq_file)
print("\n=== Table 2: Paired Per-Question Outcomes (Sample) ===")
display(df_pq[[
    "example_id", "search_budget", "method", "exact_match",
    "predicted_answer", "reference_answer", "selected_depth",
    "search_nfes", "completion_nfes", "total_nfes",
    "tree_nfes", "temporary_candidate_nfes", "adalook_extra_nfes"
]].head(20))

print("\nTiming note: search time is apportioned by budget; cached completions take zero recorded seconds.")
print("NFEs report logical per-condition cost, including cached completion work.")

print("\n=== Paired changes: AdaLook vs its own IG baseline ===")
paired_rows = []
for budget in BUDGETS:
    for baseline, enhanced in (("medal", "medal_adalook"), ("ab_medal", "ab_medal_adalook")):
        base = df_pq[(df_pq.method == baseline) & (df_pq.search_budget == budget)]
        ada = df_pq[(df_pq.method == enhanced) & (df_pq.search_budget == budget)]
        pairs = base.merge(ada, on="example_id", suffixes=("_ig", "_ada"), validate="one_to_one")
        before = pairs["exact_match_ig"].astype(str).str.lower().eq("true")
        after = pairs["exact_match_ada"].astype(str).str.lower().eq("true")
        paired_rows.append(dict(search_budget=budget, baseline=baseline, variant=enhanced,
            paired_questions=len(pairs), wrong_to_right=int((~before & after).sum()),
            right_to_wrong=int((before & ~after).sum()),
            both_correct=int((before & after).sum()), both_wrong=int((~before & ~after).sum())))
df_pairs = pd.DataFrame(paired_rows)
display(df_pairs)
df_pairs.to_csv(OUTPUT_DIR / "summaries" / f"e1_{STAGE}_paired.csv", index=False)


## 11. Run Offline E3 Analysis & Display Publication Figures
Run 100% offline tree and compute-allocation analysis directly on saved E1 traces, generating summary metrics and all 7 publication figures.

In [ ]:
from dlm_project.tasks.gsm8k.e3_analysis import analyze_e3_stage
from IPython.display import Image, display, Markdown

print("Running E3 Offline Tree Allocation & Topology Analysis...")
e3_q, e3_agg = analyze_e3_stage(
    results_dir=OUTPUT_DIR / STAGE,
    output_dir=OUTPUT_DIR / "summaries",
    stage=STAGE,
    methods=tuple(METHODS),
    budgets=tuple(BUDGETS),
)
print("✓ E3 Analysis Completed successfully.")

figures_dir = OUTPUT_DIR / "summaries" / "e3_figures"
fig_titles = {
    "fig1_nfe_by_depth": "Figure 1: Real-Tree NFE Allocation by Depth (excludes temporary AdaLook evaluations) — All four variants across budgets",
    "fig2_branching_factors": "Figure 2: Branching Factor Distribution — Children per expanded parent",
    "fig3_candidate_density": "Figure 3: Candidate Density vs Depth — Minimum-C availability and selection depth",
    "fig4_root_subtrees": "Figure 4: Root-Subtree Compute Concentration — Real-tree effort across root branches",
    "fig5_reward_redundancy": "Figure 5: Reward and Search Redundancy — Negative IG rates and duplicate states",
    "fig6_accuracy_vs_budget": "Figure 6: Accuracy vs Search Budget — Performance scaling curve",
    "fig7_compute_breakdown": "Figure 7: Search Compute — Real tree, temporary comparisons, and rollout",
}

for fig_key, title in fig_titles.items():
    png_path = figures_dir / f"{fig_key}.png"
    if png_path.is_file():
        display(Markdown(f"### {title}"))
        display(Image(filename=str(png_path)))


## 12. Interactive Search Tree & Decision Trace Inspector
Examine candidate shortlists, selection gap $\Delta_d = d_{\max} - d^*$, and Student-t GEN/CONT posterior traces for any question.

In [ ]:
import json
from pathlib import Path

INSPECT_QUESTION_IDX = 0
INSPECT_BUDGET = 256

# Ensure Google Drive is mounted if running in Colab
if Path("/content").is_dir() and not Path("/content/drive/MyDrive").is_dir():
    print("[Notice] Mounting Google Drive to access saved results...")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print(f"  Could not auto-mount Google Drive: {e}")

# Resolve the actual results root directory
search_roots = [
    globals().get("OUTPUT_DIR"),
    Path("/content/drive/MyDrive/CSCI544-DLM/results/adalook_eval_70"),
    Path("/content/CSCI544-DLM/.cache/e1_e3/results/adalook_eval_70"),
    globals().get("REPO_DIR", Path.cwd()) / ".cache/e1_e3/results/adalook_eval_70",
    Path.cwd() / "results/adalook_eval_70",
]
actual_output_dir = next(
    (p for p in search_roots if p is not None and (p / STAGE).is_dir()),
    globals().get("OUTPUT_DIR", Path("results/adalook_eval_70"))
)

q_info = ctx["examples"][INSPECT_QUESTION_IDX]
ex_id = q_info["example_id"]
print(f"=== Inspecting Question {ex_id} (Budget {INSPECT_BUDGET}) ===")
print("Question:\n", q_info["question"])
print("Reference Answer:", q_info.get("reference_solution", "").split("####")[-1].strip())
print(f"Results Directory: {actual_output_dir / STAGE} (exists: {(actual_output_dir / STAGE).exists()})")

for m in METHODS:
    chk_dir = actual_output_dir / STAGE / m / ex_id / "checkpoints" / str(INSPECT_BUDGET)
    sel_path = chk_dir / "selection.json"
    met_path = chk_dir / "metrics.json"
    tree_path = chk_dir / "tree.json"

    if sel_path.is_file() or met_path.is_file():
        sel_record = json.loads(sel_path.read_text()) if sel_path.is_file() else {}
        met_record = json.loads(met_path.read_text()) if met_path.is_file() else {}

        selected_id = sel_record.get("selected_node_id", met_record.get("selected_node_id"))
        selected_depth = sel_record.get("selected_depth", met_record.get("selected_depth"))

        selected_q = met_record.get("selected_q", sel_record.get("selected_q"))
        if selected_q is None and "node_statistics" in sel_record:
            selected_q = next((s["q_value"] for s in sel_record["node_statistics"] if s["node_id"] == selected_id), None)

        max_depth = met_record.get("max_depth_reached", sel_record.get("max_depth_reached", "N/A"))
        gap = met_record.get("selection_depth_gap", sel_record.get("selection_depth_gap", "N/A"))

        distinct_by_depth = met_record.get("distinct_candidates_by_depth") or sel_record.get("distinct_candidates_by_depth")
        if not distinct_by_depth and tree_path.is_file():
            tree_data = json.loads(tree_path.read_text())
            counts = {}
            for n in tree_data.get("nodes", []):
                counts.setdefault(n["depth"], set()).add(n.get("state_hash", n["node_id"]))
            distinct_by_depth = {d: len(s) for d, s in sorted(counts.items())}

        shortlist_ids = sel_record.get("shortlist_ids") or met_record.get("shortlist_node_ids", [])

        print(f"\n--- {m.upper()} Selection Record ---")
        print(f"  Selected Node: {selected_id} | Depth: {selected_depth}")
        print(f"  Selected Q: {selected_q:.4f}" if selected_q is not None else "  Selected Q: N/A")
        print(f"  Max Depth Reached: {max_depth} | Selection Gap (Δd): {gap}")
        print(f"  Distinct Candidates by Depth: {distinct_by_depth}")
        print(f"  Shortlist Node IDs: {shortlist_ids}")
        print("  Reward mode:", met_record.get("reward_mode"))
        print("  NFE (tree / comparison / rollout):", met_record.get("tree_nfes"),
              met_record.get("temporary_candidate_nfes"), met_record.get("adalook_extra_nfes"))
    else:
        print(f"\n--- {m.upper()} Selection Record ---")
        print(f"  [Not Found] Expected path: {sel_path}")
        parent = chk_dir.parent
        if not parent.exists():
            print(f"  Parent directory does not exist: {parent}")
        elif parent.is_dir():
            print(f"  Available checkpoints for {ex_id}: {[p.name for p in parent.iterdir()]}")

# Both AB variants use GEN/CONT. Filter the full-run trace to the inspected budget.
for method in ("ab_medal", "ab_medal_adalook"):
    if method not in METHODS:
        continue
    dec_path = actual_output_dir / STAGE / method / ex_id / "decisions.jsonl"
    print(f"\n--- {method.upper()} Decision Trace Sample (through budget {INSPECT_BUDGET}) ---")
    if dec_path.is_file():
        events = [json.loads(line) for line in dec_path.read_text().splitlines()]
        events = [d for d in events if d.get("search_model_calls", 0) < INSPECT_BUDGET]
        for d in events[:10]:
            print(f"  Cycle {d.get('cycle')}: parent node {d.get('node_id')} -> "
                  f"{d.get('choice', d.get('decision_type'))} (forced={d.get('forced')})")
    else:
        print(f"  [Not Found] {dec_path}")


## 13. Synchronize All Artifacts to Google Drive
Explicitly flush and sync all in-memory filesystem buffers to Google Drive cloud storage so all traces, trees, completions, CSVs, and plots are safely persisted and reconstructible even if Colab disconnects.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
    drive.mount("/content/drive")
    print("✓ Google Drive synchronization complete. All artifacts safely flushed to Drive.")
else:
    print(f"✓ Local artifacts safely saved at: {OUTPUT_DIR}")